# Ingest taxonomy summaries → BERDL lakehouse (`nmdc_results` namespace)

Loads the parquet files produced by `fetch_taxonomy_summaries.ipynb` into the
`nmdc` tenant's lakehouse as the `nmdc_results` namespace — the shared landing
spot for this set of "data object" loaders. Future loaders that produce more
data-object-derived tables can ingest into the same namespace without
overwriting unrelated tables (each table is overwritten/appended independently).

On-pod, so no SSH tunnels / proxy / `berdl-remote spawn` — Spark Connect, MinIO,
and `data_lakehouse_ingest.ingest()` are all auto-configured by the kernel.

Two-phase flow (matching BERIL convention):
1. **Upload** local parquets to MinIO bronze under
   `tenant-general-warehouse/nmdc/datasets/results/`.
2. **Ingest** registers Delta tables under
   `s3a://cdm-lake/tenant-sql-warehouse/nmdc/nmdc_results.db/`.

After ingest, query as e.g.
`SELECT * FROM nmdc_results.checkm_statistics LIMIT 10`.

### Configuration

In [ ]:
import os
from pathlib import Path

# ── User configuration ────────────────────────────────────────────────────
TENANT      = "nmdc"
DATASET     = "results"
BUCKET      = "cdm-lake"
MODE        = "overwrite"  # "overwrite" or "append"

# Where the source parquets live (output of fetch_taxonomy_summaries.ipynb)
SOURCE_DIR  = Path(os.environ.get("TAXONOMY_OUT_DIR", "loaded_taxonomy"))
# ──────────────────────────────────────────────────────────────────────────

NAMESPACE       = f"{TENANT}_{DATASET}"
BRONZE_PREFIX   = f"tenant-general-warehouse/{TENANT}/datasets/{DATASET}"
BRONZE_BASE     = f"s3a://{BUCKET}/{BRONZE_PREFIX}"
SILVER_BASE     = f"s3a://{BUCKET}/tenant-sql-warehouse/{TENANT}/{NAMESPACE}.db"

print(f"Tenant       : {TENANT}")
print(f"Dataset      : {DATASET}")
print(f"Namespace    : {NAMESPACE}")
print(f"Mode         : {MODE}")
print(f"Source dir   : {SOURCE_DIR.resolve()}")
print(f"Bronze base  : {BRONZE_BASE}")
print(f"Silver base  : {SILVER_BASE}")

### Setup — Spark Connect + MinIO

`get_spark_session` and `get_minio_client` are auto-imported by the BERDL kernel
startup script. Pass `tenant_name=TENANT` so Spark writes tables to the nmdc
warehouse instead of the user's personal one.

In [ ]:
spark = get_spark_session(app_name=f"ingest_{NAMESPACE}", tenant_name=TENANT)
mc    = get_minio_client()
print(f"Spark version: {spark.version}")
try:
    print(f"MinIO endpoint: {mc._base_url._url.netloc}")
except AttributeError:
    print("MinIO endpoint: (endpoint not directly inspectable)")

### Inputs — locate source parquets

We ingest one Delta table per parquet. Each table name is the parquet file's stem.
Files that don't exist (e.g. `gtdbtk_archaeal_summary.parquet`, which the loader
skipped because every input was a placeholder) are dropped from the ingest plan.

In [ ]:
EXPECTED_TABLES = [
    "checkm_statistics",
    "gottcha2_classification_report",
    "gtdbtk_archaeal_summary",
    "gtdbtk_bacterial_summary",
    "kraken2_classification_report",
]

plan = []
for name in EXPECTED_TABLES:
    local = SOURCE_DIR / f"{name}.parquet"
    if not local.exists():
        print(f"  -- {name}: missing on disk, will skip")
        continue
    size_mb = local.stat().st_size / 1024 / 1024
    plan.append({"name": name, "local": local, "size_mb": size_mb})
    print(f"  OK {name}: {size_mb:,.1f} MB")

if not plan:
    raise RuntimeError(f"No source parquets found under {SOURCE_DIR.resolve()}")
print(f"\n{len(plan)} table(s) will be ingested.")

### Phase 1 — upload to MinIO bronze

Spark executors run on other cluster nodes and can't see the notebook's local
filesystem, so we upload via the **driver-side MinIO client** instead of
`spark.read.parquet(file://...)`. Each parquet lands at
`{BRONZE_BASE}/<table>.parquet` (i.e.
`s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/results/<table>.parquet`).

In [ ]:
import re
import time
import pyarrow.parquet as pq

# Delta Lake forbids column names containing spaces, commas, semicolons, braces, parens,
# newline, tab, or equals. The GTDBTK Bacterial parquet has columns like
# "other_related_references(genome_id,species_name,radius,ANI,AF)" — replace illegal
# chars with underscores before upload so ingest doesn't silently fail on that table.
_ILLEGAL_COL_CHARS_RE = re.compile(r"[ ,;{}()\n\t=]+")


def _sanitize_col(name: str) -> str:
    return _ILLEGAL_COL_CHARS_RE.sub("_", name).strip("_")


def _sanitize_parquet(local_path: Path) -> Path:
    """If column names are Delta-incompatible, rewrite to a sibling .sanitized.parquet
    and return that path; otherwise return the original."""
    schema    = pq.read_schema(local_path)
    original  = schema.names
    sanitized = [_sanitize_col(c) for c in original]
    if original == sanitized:
        return local_path
    renamed = [(o, s) for o, s in zip(original, sanitized) if o != s]
    table = pq.read_table(local_path)
    table = table.rename_columns(sanitized)
    fixed = local_path.with_suffix(".sanitized.parquet")
    pq.write_table(table, fixed)
    print(f"    sanitized {len(renamed)} column name(s): {renamed}")
    return fixed


for entry in plan:
    name        = entry["name"]
    local_path  = entry["local"].resolve()
    upload_path = _sanitize_parquet(local_path)
    object_key  = f"{BRONZE_PREFIX}/{name}.parquet"
    bronze_uri  = f"s3a://{BUCKET}/{object_key}"
    print(f"  uploading {name} ({entry['size_mb']:,.1f} MB) → {bronze_uri}")
    t0 = time.monotonic()
    mc.fput_object(BUCKET, object_key, str(upload_path))
    print(f"    done in {time.monotonic() - t0:.1f}s")
    entry["bronze"]     = bronze_uri
    entry["object_key"] = object_key

print("\nAll uploads complete.")

### Phase 2 — build the ingest config

Schema is inferred per table from parquet metadata (no `schema_sql` needed).
Each table's `bronze_path` is a relative path under `paths.bronze_base`.

In [ ]:
config = {
    "tenant":  TENANT,
    "dataset": DATASET,
    "paths": {
        "bronze_base": BRONZE_BASE,
        "silver_base": SILVER_BASE,
    },
    "defaults": {
        "parquet": {},
    },
    "tables": [
        {
            "name":         entry["name"],
            "format":       "parquet",
            "bronze_path":  f"{entry['name']}.parquet",   # joined with bronze_base by ConfigLoader
            "write_mode":   MODE,
        }
        for entry in plan
    ],
}

import json
print(json.dumps(config, indent=2))

### Phase 3 — run ingest

`ingest()` reads parquet from each `bronze_path`, applies any schema/transforms,
and writes Delta tables to `silver_base`. The namespace (Hive database) is
created if it doesn't exist.

In [ ]:
report = ingest(config=config, spark=spark, minio_client=mc)

import json
print(json.dumps(report, indent=2, default=str))

### Verify — read back from the lakehouse

Round-trip check: list the namespace, then read each Delta table via SQL and
confirm the row count matches the source parquet.

In [ ]:
import pyarrow.parquet as pq

print(f"=== tables in namespace `{NAMESPACE}` ===")
spark.sql(f"SHOW TABLES IN {NAMESPACE}").show(truncate=False)

print(f"\n=== row counts: lakehouse vs source ===")
for entry in plan:
    name = entry["name"]
    delta_count = spark.sql(f"SELECT COUNT(*) AS n FROM {NAMESPACE}.{name}").collect()[0]["n"]
    # Read row count from local parquet metadata only (no executor needed)
    src_count   = pq.read_metadata(str(entry["local"])).num_rows
    match       = "OK" if delta_count == src_count else "MISMATCH"
    print(f"  {match:9s} {name}: delta={delta_count:>12,}  source={src_count:>12,}")

In [ ]:
# Spot-check: schema + first rows of one table
spark.sql(f"DESCRIBE {NAMESPACE}.gottcha2_classification_report").show(truncate=False)
spark.sql(f"SELECT * FROM {NAMESPACE}.gottcha2_classification_report LIMIT 5").show(truncate=False)